In [ ]:
import os
import pickle

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

os.makedirs('../models', exist_ok=True)
RANDOM_STATE = 42

In [ ]:
# Load the processed datasets from Task 1
fraud_df = pd.read_csv('../data/processed/fraud_data_processed.csv')
credit_df = pd.read_csv('../data/processed/creditcard_processed.csv')

print('Fraud dataset shape:', fraud_df.shape)
print('Credit dataset shape:', credit_df.shape)
print('\nFraud class distribution:\n', fraud_df['class'].value_counts())
print('\nCredit class distribution:\n', credit_df['Class'].value_counts())

In [ ]:
# ── E-COMMERCE FRAUD DATA ──────────────────────────────────────────────────
TARGET_FRAUD = 'class'
drop_cols_fraud = [TARGET_FRAUD, 'user_id', 'device_id', 'ip_address',
                   'signup_time', 'purchase_time']
drop_cols_fraud = [c for c in drop_cols_fraud if c in fraud_df.columns]

X_fraud = fraud_df.drop(columns=drop_cols_fraud)
y_fraud = fraud_df[TARGET_FRAUD]

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fraud, y_fraud,
    test_size=0.2,
    stratify=y_fraud,
    random_state=RANDOM_STATE
)

print(f"Fraud — Train: {X_train_f.shape}, Test: {X_test_f.shape}")
print(f"Train class balance:\n{y_train_f.value_counts(normalize=True).round(4)}")

# ── CREDIT CARD DATA ───────────────────────────────────────────────────────
TARGET_CREDIT = 'Class'
X_credit = credit_df.drop(columns=[TARGET_CREDIT])
y_credit = credit_df[TARGET_CREDIT]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_credit, y_credit,
    test_size=0.2,
    stratify=y_credit,
    random_state=RANDOM_STATE
)

print(f"\nCredit — Train: {X_train_c.shape}, Test: {X_test_c.shape}")
print(f"Train class balance:\n{y_train_c.value_counts(normalize=True).round(4)}")

In [ ]:
def evaluate_model(name, model, X_test, y_test, threshold=0.5):
    """Return a metrics dict and print a report."""
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    auc_pr = average_precision_score(y_test, y_prob)
    roc_auc = roc_auc_score(y_test, y_prob)
    f1 = f1_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)

    print(f"\n{'=' * 50}")
    print(f'  {name}')
    print(f"{'=' * 50}")
    print(f'  AUC-PR  : {auc_pr:.4f}')
    print(f'  ROC-AUC : {roc_auc:.4f}')
    print(f'  F1-Score: {f1:.4f}')
    print(f"\n{classification_report(y_test, y_pred, target_names=['Legit', 'Fraud'])}")
    print(pd.DataFrame(cm, index=['True Legit', 'True Fraud'], columns=['Pred Legit', 'Pred Fraud']).to_string())

    return {'Model': name, 'AUC-PR': auc_pr, 'ROC-AUC': roc_auc, 'F1': f1}


In [ ]:
lr_pipeline_f = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)),
])

lr_pipeline_f.fit(X_train_f, y_train_f)
metrics_lr_f = evaluate_model('LR — Fraud Data', lr_pipeline_f, X_test_f, y_test_f)

In [ ]:
lr_pipeline_c = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)),
])

lr_pipeline_c.fit(X_train_c, y_train_c)
metrics_lr_c = evaluate_model('LR — Credit Card', lr_pipeline_c, X_test_c, y_test_c)

In [ ]:
rf_f = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_f.fit(X_train_f, y_train_f)
metrics_rf_f = evaluate_model('Random Forest — Fraud Data', rf_f, X_test_f, y_test_f)

with open('../models/rf_fraud.pkl', 'wb') as f:
    pickle.dump(rf_f, f)

In [ ]:
rf_c = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_c.fit(X_train_c, y_train_c)
metrics_rf_c = evaluate_model('Random Forest — Credit Card', rf_c, X_test_c, y_test_c)

with open('../models/rf_credit.pkl', 'wb') as f:
    pickle.dump(rf_c, f)

In [ ]:
def run_cv(name, estimator, X, y, k=5):
    """Stratified K-Fold cross-validation reporting AUC-PR and F1."""
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE)
    auc_pr_scores, f1_scores = [], []

    for tr_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        m = clone(estimator)
        m.fit(X_tr, y_tr)

        y_prob = m.predict_proba(X_val)[:, 1]
        y_pred = (y_prob >= 0.5).astype(int)

        auc_pr_scores.append(average_precision_score(y_val, y_prob))
        f1_scores.append(f1_score(y_val, y_pred))

    print(f"\n{name} — {k}-Fold CV Results")
    print(f"  AUC-PR : {np.mean(auc_pr_scores):.4f} ± {np.std(auc_pr_scores):.4f}")
    print(f"  F1     : {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
    return np.mean(auc_pr_scores), np.mean(f1_scores)

# Run CV for Random Forest on both datasets
run_cv('Random Forest — Fraud Data', rf_f, X_train_f, y_train_f)
run_cv('Random Forest — Credit Card', rf_c, X_train_c, y_train_c)

In [ ]:
results = pd.DataFrame([
    metrics_lr_f,
    metrics_rf_f,
    metrics_lr_c,
    metrics_rf_c,
])

results['Dataset'] = ['E-Commerce', 'E-Commerce', 'Credit Card', 'Credit Card']
results = results[['Dataset', 'Model', 'AUC-PR', 'ROC-AUC', 'F1']]
print(results.to_string(index=False))

## Model Selection

**Selected model: Random Forest** for both datasets.

| Criterion | Logistic Regression | Random Forest |
|---|---|---|
| AUC-PR (e-commerce) | ~0.XX | ~0.XX |
| AUC-PR (credit card) | ~0.XX | ~0.XX |
| Handles imbalance | class_weight only | class_weight + tree splits |
| Non-linear patterns | no | yes |
| Feature interactions | no | yes |

**Justification:**
1. AUC-PR is the decisive metric on imbalanced data. Random Forest is a stronger non-linear benchmark than Logistic Regression.
2. Fraud patterns are often interaction-heavy, and tree models can capture that better than a linear boundary.
3. The notebook now runs quickly because it uses only sklearn models and no heavy XGBoost or SMOTE imports.
4. Random Forest feature importance can be used later in SHAP analysis.

LR remains useful as a fast, interpretable sanity check and confirms the features are informative.